# Data & RAG — Built From Scratch

This notebook builds the entire data and RAG pipeline from scratch using raw libraries — no imports from `fin_agents`.

**What we build:**
1. Fetch company fundamentals with `yfinance`
2. Fetch news headlines with `requests` + NewsAPI
3. Fetch SEC filings from EDGAR with `requests` + `BeautifulSoup`
4. Chunk the filing text with `RecursiveCharacterTextSplitter`
5. Embed chunks with `sentence_transformers`
6. Store in `ChromaDB`
7. Query (retrieve) relevant chunks by semantic similarity

---
## 0 · Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

TICKER = 'AAPL'
print('Ready')

Ready


---
## 1 · Fundamentals — yfinance

`yfinance` wraps the Yahoo Finance API. `Ticker.info` returns a dict with ~100 fields.

In [2]:
import yfinance as yf

t = yf.Ticker(TICKER)
info = t.info

# Pull only what we care about
name       = info.get('longName', TICKER)
sector     = info.get('sector')
market_cap = info.get('marketCap')
pe_ratio   = info.get('trailingPE')
eps        = info.get('trailingEps')
revenue    = info.get('totalRevenue')
summary    = info.get('longBusinessSummary', '')

print(f'Company    : {name} ({TICKER})')
print(f'Sector     : {sector}')
print(f'Market Cap : ${market_cap:,.0f}')
print(f'P/E        : {pe_ratio}')
print(f'EPS        : {eps}')
print(f'Revenue TTM: ${revenue:,.0f}')
print(f'\nBusiness Summary:')
print(summary[:400])

Company    : Apple Inc. (AAPL)
Sector     : Technology
Market Cap : $3,918,759,460,864
P/E        : 33.79213
EPS        : 7.89
Revenue TTM: $435,617,005,568

Business Summary:
Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple brande


---
## 2 · News — NewsAPI

Plain `requests.get` to `newsapi.org/v2/everything`. Returns up to 100 articles sorted by date.

In [3]:
import requests
from datetime import datetime

response = requests.get(
    'https://newsapi.org/v2/everything',
    params={
        'q': TICKER,
        'sortBy': 'publishedAt',
        'pageSize': 5,
        'language': 'en',
    },
    headers={'X-Api-Key': os.environ['NEWSAPI_KEY']},
    timeout=15,
)
response.raise_for_status()
articles = response.json().get('articles', [])

print(f'Found {len(articles)} articles\n')
for i, a in enumerate(articles, 1):
    published = datetime.fromisoformat(a['publishedAt'].replace('Z', '+00:00'))
    print(f'[{i}] {a["title"]}')
    print(f'     Source   : {a["source"]["name"]}')
    print(f'     Published: {published.date()}')
    print(f'     URL      : {a["url"]}')
    print()

Found 5 articles

[1] 2 charts show why Magnificent 7 stocks are being loved again
     Source   : Yahoo Entertainment
     Published: 2026-04-20
     URL      : https://finance.yahoo.com/news/2-charts-show-why-magnificent-7-stocks-are-being-loved-again-151105348.html

[2] One Analyst Breaks From The Herd, Says Apple’s DRAM Buying Spree Won’t Crush Margins Despite 2.4 Exabyte Appetite
     Source   : Wccftech
     Published: 2026-04-20
     URL      : https://wccftech.com/goldman-sachs-breaks-from-wall-street-panic-says-apples-dram-hoarding-wont-crush-margins-despite-2-4-exabyte-appetite/

[3] Anker Japan、単ポート最大140W出力でMacBook Proの高速充電も可能なモバイルバッテリー「Anker Prime Power Bank (20100mAh, 220W)」と専用充電ベースのセットモデルを発売。
     Source   : Applech2.com
     Published: 2026-04-20
     URL      : https://applech2.com/archives/20260420-anker-prime-power-bank-20k-220w-with-base.html

[4] The S&P 500 has blown past 7,000 in an epic comeback rally. Here’s why it can keep going.
     Source   : MarketWatch
   

---
## 3 · SEC Filing — EDGAR

EDGAR has a public JSON API. We:
1. Resolve ticker → CIK (company identifier)
2. Fetch the submissions list to find the latest 10-K or 10-Q
3. Download the HTML filing and strip to plain text with BeautifulSoup

In [4]:
# Step 1 — resolve ticker to CIK
HEADERS = {'User-Agent': os.environ['SEC_USER_AGENT']}

tickers_json = requests.get(
    'https://www.sec.gov/files/company_tickers.json',
    headers=HEADERS, timeout=15
).json()

cik = None
for row in tickers_json.values():
    if row['ticker'].upper() == TICKER.upper():
        cik = str(row['cik_str']).zfill(10)
        break

print(f'CIK for {TICKER}: {cik}')

CIK for AAPL: 0000320193


In [5]:
# Step 2 — find the latest 10-K or 10-Q
subs = requests.get(
    f'https://data.sec.gov/submissions/CIK{cik}.json',
    headers=HEADERS, timeout=15
).json()

forms = subs['filings']['recent']

filing_type = None
filing_date = None
filing_url  = None

for i, form in enumerate(forms['form']):
    if form in ('10-K', '10-Q'):
        acc = forms['accessionNumber'][i].replace('-', '')
        doc = forms['primaryDocument'][i]
        filing_type = form
        filing_date = forms['filingDate'][i]
        filing_url  = f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc}/{doc}'
        break

print(f'Filing type: {filing_type}')
print(f'Filing date: {filing_date}')
print(f'URL        : {filing_url}')

Filing type: 10-Q
Filing date: 2026-01-30
URL        : https://www.sec.gov/Archives/edgar/data/320193/000032019326000006/aapl-20251227.htm


In [6]:
# Step 3 — download HTML and strip to plain text
from bs4 import BeautifulSoup

html = requests.get(filing_url, headers=HEADERS, timeout=30).text
text = BeautifulSoup(html, 'html.parser').get_text(separator=' ', strip=True)

print(f'Raw HTML size  : {len(html):,} chars')
print(f'Plain text size: {len(text):,} chars')
print(f'\nFirst 500 chars:')
print('-' * 60)
print(text[:500])

Raw HTML size  : 765,973 chars
Plain text size: 66,160 chars

First 500 chars:
------------------------------------------------------------
aapl-20251227 false 2026 Q1 0000320193 --09-26 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent P406D P403D xbrli:shares iso4217:USD iso4217:USD xbrli:shares xbrli:pure aapl:Customer aapl:Vendor 0000320193 2025-09-28 2025-12-27 0000320193 us-gaap:CommonStockMember 2025-09-28 2025-12-27 0000320193 aapl:A1.625NotesDue2026Member 2025-09-28 2025-12-27 0000320193 aapl:A2.000NotesDue2027Member 2025-09-28 2025-12-27 0000320193 aapl:


---
## 4 · Chunking

`RecursiveCharacterTextSplitter` splits text by paragraphs → sentences → words, trying to keep semantic units together. `chunk_overlap` ensures context isn't lost at boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

chunks = splitter.split_text(text)

print(f'Total chunks : {len(chunks)}')
print(f'Avg chunk len: {sum(len(c) for c in chunks) // len(chunks)} chars')
print(f'\n--- Chunk 0 ---')
print(chunks[0])
print(f'\n--- Chunk 1 (notice overlap with chunk 0) ---')
print(chunks[1][:300])

---
## 5 · Embeddings

An embedding converts text into a vector of numbers. Similar texts produce similar vectors. `BAAI/bge-m3` is a state-of-the-art free model with 8192-token context — ideal for long financial text.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-m3')

# Embed two chunks to see what a vector looks like
sample_chunks = chunks[:2]
embeddings = model.encode(sample_chunks, show_progress_bar=True)

print(f'\nEmbedding shape : {embeddings.shape}')  # (2, 1024)
print(f'Vector dims     : {embeddings.shape[1]}')
print(f'First 5 values  : {embeddings[0][:5]}')

# Cosine similarity between the two chunks
from numpy.linalg import norm
import numpy as np
cos_sim = np.dot(embeddings[0], embeddings[1]) / (norm(embeddings[0]) * norm(embeddings[1]))
print(f'\nCosine similarity (chunk 0 vs chunk 1): {cos_sim:.4f}')
print('(1.0 = identical meaning, 0.0 = unrelated)')

---
## 6 · Store in ChromaDB

ChromaDB is a vector database. We store each chunk with:
- its text (document)
- its embedding (handled automatically by the embedding function)
- metadata (ticker, filing type, date, chunk index)
- a stable ID (so re-running doesn't create duplicates)

In [ ]:
import hashlib
import chromadb
from chromadb.utils import embedding_functions

# Connect to (or create) a persistent DB on disk
client = chromadb.PersistentClient(path='../chroma_db_scratch')

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='BAAI/bge-m3'
)

col = client.get_or_create_collection(
    name='filings_scratch',
    embedding_function=embed_fn,
)

# Build stable IDs: sha256(TICKER_filingDate_chunkIndex)
def chunk_id(ticker, filing_date, idx):
    key = f'{ticker.upper()}_{filing_date}_{idx}'
    return hashlib.sha256(key.encode()).hexdigest()[:16]

ids       = [chunk_id(TICKER, filing_date, i) for i in range(len(chunks))]
metadatas = [
    {'ticker': TICKER, 'filing_type': filing_type, 'filing_date': filing_date, 'chunk_index': i}
    for i in range(len(chunks))
]

col.upsert(ids=ids, documents=chunks, metadatas=metadatas)

print(f'Stored {len(chunks)} chunks for {TICKER}')
print(f'Collection total: {col.count()} chunks')

---
## 7 · Retrieve — Semantic Search

To query, we embed the question and find the chunks whose vectors are closest (smallest cosine distance). We can also filter by metadata (e.g. only return chunks for AAPL).

In [ ]:
query = 'What are the main risks disclosed in the filing?'

results = col.query(
    query_texts=[query],
    n_results=3,
    where={'ticker': TICKER},
)

print(f'Query: {query}\n')
for i, (doc, meta, dist) in enumerate(
    zip(results['documents'][0], results['metadatas'][0], results['distances'][0]), 1
):
    print(f'--- Result {i} (distance: {dist:.4f}) ---')
    print(f'Date: {meta["filing_date"]}  |  Type: {meta["filing_type"]}  |  Chunk: {meta["chunk_index"]}')
    print(doc[:400])
    print()

In [ ]:
# Try a different query to see how results change
query2 = 'What was the revenue and gross margin this quarter?'

results2 = col.query(
    query_texts=[query2],
    n_results=3,
    where={'ticker': TICKER},
)

print(f'Query: {query2}\n')
for i, (doc, dist) in enumerate(
    zip(results2['documents'][0], results2['distances'][0]), 1
):
    print(f'--- Result {i} (distance: {dist:.4f}) ---')
    print(doc[:400])
    print()